# 04. Full Agent Integration — 멀티모달 분석 파이프라인

## 학습 목표
1. Notebook 02(Vision) + Notebook 03(RAG)를 하나의 파이프라인으로 통합
2. Orchestrator → Vision Analyst → RAG Retriever → Report Writer 4-에이전트 협력 구현
3. LangGraph `send` API 및 병렬 서브그래프 패턴 이해
4. Streaming 응답 처리

## 전체 아키텍처
```
사용자 입력 (이미지 + 질문)
          ↓
   [Orchestrator]  ← 라우팅 결정 (vision / rag / both)
    ↙         ↘
[Vision      [RAG
 Analyst]    Retriever]  ← ChromaDB
    ↘         ↙
  [Report Writer]  ← Vision 결과 + RAG 컨텍스트 통합
          ↓
  구조화된 분석 리포트
```

In [ ]:
# ── 한글 폰트 & 환경 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

print('환경 설정 완료')

In [ ]:
# -- 핵심 임포트
import os, json, base64, time, re
from pathlib import Path
from typing import TypedDict, List, Optional, Literal
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from PIL import Image
import io, requests

load_dotenv('../.env')

# Ollama 모델 설정
OLLAMA_LLM_MODEL = "llama3.2"

def check_ollama_model(model):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        models = [m["name"] for m in r.json().get("models", [])]
        return any(model.split(":")[0] in m for m in models)
    except:
        return False

USE_OLLAMA = check_ollama_model(OLLAMA_LLM_MODEL)
USE_CLAUDE = False  # Ollama 모드

if USE_OLLAMA:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, temperature=0)
    print(f"Ollama ({OLLAMA_LLM_MODEL}): 사용 가능")
else:
    llm = None
    print("Ollama 없음 → Mock 모드")

print("임포트 완료")


---
## Part 1. 공유 유틸리티 & CV 래퍼 재정의

Notebook 02에서 사용한 CV 래퍼와 이미지 유틸리티를 이 노트북에서 재사용합니다.

In [ ]:
# ── 이미지 유틸리티

USE_REAL_MODELS = False  # True: 실제 CV 모델, False: Mock

def image_to_base64(img: Image.Image, fmt: str = 'JPEG') -> str:
    """PIL Image → base64 문자열"""
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

def load_or_create_test_image(path: str = None) -> tuple:
    """테스트 이미지 로드 또는 생성, (image, base64) 반환"""
    if path and os.path.exists(path):
        img = Image.open(path).convert('RGB').resize((640, 480))
        print(f'이미지 로드: {path}')
    else:
        # Mock 이미지 생성 (도로 장면 시뮬레이션)
        import numpy as np
        arr = np.zeros((480, 640, 3), dtype=np.uint8)
        arr[240:, :] = [80, 80, 80]   # 도로 (회색)
        arr[:240, :] = [135, 206, 235]  # 하늘 (하늘색)
        arr[200:280, 100:200] = [0, 0, 200]   # 차량 1 (파랑)
        arr[210:270, 400:520] = [200, 0, 0]   # 차량 2 (빨강)
        arr[160:240, 280:340] = [0, 180, 0]   # 신호등 (초록)
        img = Image.fromarray(arr)
        print('Mock 테스트 이미지 생성 (도로 장면)')
    return img, image_to_base64(img)

test_img, test_img_b64 = load_or_create_test_image('../data/test_images/sample.jpg')
print(f'이미지 크기: {test_img.size}, base64 길이: {len(test_img_b64)}')

In [ ]:
# ── CV 래퍼 (Notebook 02 재활용)
import random

class YOLOWrapper:
    """YOLOv8 Mock 래퍼"""
    def detect(self, image_b64: str) -> dict:
        if USE_REAL_MODELS:
            pass  # 실제 모델 로직 (Notebook 02 참조)
        # Mock 결과
        objects = [
            {'class': 'car',    'confidence': 0.92, 'bbox': [100, 200, 200, 280]},
            {'class': 'car',    'confidence': 0.87, 'bbox': [400, 210, 520, 270]},
            {'class': 'traffic light', 'confidence': 0.78, 'bbox': [280, 160, 340, 240]},
            {'class': 'road',   'confidence': 0.95, 'bbox': [0, 240, 640, 480]},
        ]
        return {
            'objects': objects,
            'count': len(objects),
            'classes': list({o['class'] for o in objects}),
            'model': 'yolov8n-mock'
        }

class SAMWrapper:
    """SAM Mock 래퍼"""
    def segment(self, image_b64: str, bbox: list = None) -> dict:
        if USE_REAL_MODELS:
            pass
        return {
            'mask_count': random.randint(3, 6),
            'coverage_ratio': round(random.uniform(0.4, 0.7), 2),
            'largest_segment': {'class': 'road', 'area_ratio': 0.42},
            'model': 'sam-vit-b-mock'
        }

class DepthWrapper:
    """DepthAnything Mock 래퍼"""
    def estimate(self, image_b64: str) -> dict:
        if USE_REAL_MODELS:
            pass
        return {
            'mean_depth': round(random.uniform(5.0, 15.0), 1),
            'near_objects': ['car (2.1m)', 'traffic light (4.3m)'],
            'far_objects': ['background road (18m)'],
            'depth_range': {'min': 1.2, 'max': 25.0},
            'model': 'depth-anything-v2-mock'
        }

yolo = YOLOWrapper()
sam  = SAMWrapper()
depth = DepthWrapper()
print('CV 래퍼 초기화 완료 (Mock 모드)')

In [ ]:
# -- RAG 벡터 스토어 로드 (Notebook 03 결과물 재활용)
import requests, gc

CHROMA_DIR = "../data/chroma_db"
COLLECTION_NAME = "cv_knowledge_base"

def check_ollama() -> bool:
    """Ollama 실행 중 + nomic-embed-text 존재 여부 확인"""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code != 200:
            return False
        models = [m["name"] for m in r.json().get("models", [])]
        return any("nomic-embed-text" in m for m in models)
    except:
        return False

OLLAMA_RUNNING = check_ollama()

if OLLAMA_RUNNING:
    from langchain_ollama import OllamaEmbeddings
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
else:
    from langchain_core.embeddings.fake import FakeEmbeddings
    embeddings = FakeEmbeddings(size=768)

print(f"Ollama+nomic: {"사용" if OLLAMA_RUNNING else "없음 → FakeEmbeddings"}")

# 벡터 스토어 로드 또는 재구축
if os.path.exists(CHROMA_DIR):
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR
    )
    print(f"ChromaDB 로드 완료 — 벡터 {vectorstore._collection.count()}개")
else:
    print("ChromaDB 없음 — 다음 셀에서 재구축합니다")
    vectorstore = None


In [ ]:
# ── ChromaDB 없을 경우 인라인 재구축
# (Notebook 03을 실행했다면 이 셀은 건너뜀)

if vectorstore is None:
    from langchain_community.document_loaders import DirectoryLoader, TextLoader
    import shutil
    
    KB_DIR = Path('../data/knowledge_base')
    KB_DIR.mkdir(parents=True, exist_ok=True)
    
    # 최소 샘플 문서
    sample_docs = {
        'yolo_overview.txt': 'YOLO는 실시간 객체 탐지 모델입니다. YOLOv8은 anchor-free 방식으로 차량, 보행자, 신호등을 탐지합니다.',
        'sam_overview.txt': 'SAM (Segment Anything Model)은 포인트, 박스 프롬프트로 객체를 세그멘테이션합니다.',
        'depth_estimation.txt': 'DepthAnything은 단안 RGB 이미지에서 픽셀별 깊이값을 추정하는 Foundation Model입니다.',
        'rag_concepts.txt': 'RAG는 외부 지식 베이스를 검색하여 LLM의 답변 정확도를 높이는 패턴입니다.',
        'langgraph_concepts.txt': 'LangGraph는 에이전트를 StateGraph로 정의하는 오케스트레이션 프레임워크입니다.',
    }
    for fname, content in sample_docs.items():
        (KB_DIR / fname).write_text(content, encoding='utf-8')
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    loader = DirectoryLoader(str(KB_DIR), glob='*.txt',
                              loader_cls=TextLoader,
                              loader_kwargs={'encoding': 'utf-8'})
    chunks = splitter.split_documents(loader.load())
    
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR
    )
    print(f'ChromaDB 재구축 완료 — {vectorstore._collection.count()}개 벡터')
else:
    print('ChromaDB 이미 로드됨 — 재구축 불필요')

retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

---
## Part 2. 통합 State 설계

4개 에이전트가 공유하는 단일 State를 설계합니다.  
각 에이전트는 자신이 담당하는 필드만 업데이트합니다.

In [ ]:
# ── 통합 파이프라인 State

class PipelineState(TypedDict):
    # ── 입력
    question: str                # 사용자 질문
    image_b64: Optional[str]     # 이미지 (없으면 RAG만 실행)
    
    # ── Orchestrator 결정
    routing: str                 # 'vision_only' | 'rag_only' | 'both'
    task_plan: str               # 작업 계획 설명
    
    # ── Vision Analyst 결과
    yolo_result: dict
    sam_result: dict
    depth_result: dict
    vision_summary: str          # Vision 분석 요약 텍스트
    
    # ── RAG Retriever 결과
    rag_context: str             # 검색된 컨텍스트
    rag_sources: List[str]       # 참고 출처
    
    # ── Report Writer 결과
    report: str                  # 최종 분석 리포트
    
    # ── 메타
    step_log: List[str]          # 각 단계 로그
    error: Optional[str]

print('PipelineState 설계 완료')
print(f'총 필드: {len(PipelineState.__annotations__)}개')
for field, ftype in PipelineState.__annotations__.items():
    print(f'  {field}: {ftype}')

---
## Part 3. Orchestrator Agent

질문과 이미지 유무를 분석하여 **어떤 에이전트를 실행할지** 결정합니다.

- 이미지 있음 + CV 관련 질문 → `both`
- 이미지 없음 → `rag_only`  
- 이미지 있지만 일반 지식 질문 → `rag_only`
- 단순 이미지 분석 → `vision_only`

In [ ]:
# -- Orchestrator 노드
ORCHESTRATOR_SYSTEM = '''당신은 멀티모달 분석 시스템의 오케스트레이터입니다.
사용자의 요청을 분석하여 어떤 에이전트를 실행할지 결정합니다.

결정 규칙:
- 이미지가 있고 이미지 분석이 필요하면: routing = "both" 또는 "vision_only"
- 이미지가 없거나 지식 검색만 필요하면: routing = "rag_only"
- 이미지 분석 + 배경 지식이 모두 필요하면: routing = "both"

반드시 JSON 형식으로만 응답하세요:
{"routing": "both|vision_only|rag_only", "task_plan": "작업 계획 (한국어)"}'''

def orchestrator_node(state: PipelineState) -> dict:
    question = state["question"]
    has_image = bool(state.get("image_b64"))
    log = state.get("step_log", [])
    log.append(f'[Orchestrator] 질문: "{question}" | 이미지: {has_image}')

    if USE_OLLAMA:
        user_msg = f"질문: {question}\n이미지 존재: {has_image}\n\n라우팅을 JSON으로 결정해주세요."
        messages = [SystemMessage(content=ORCHESTRATOR_SYSTEM), HumanMessage(content=user_msg)]
        raw = llm.invoke(messages).content.strip()
        try:
            match = re.search(r'\{.*?\}', raw, re.DOTALL)
            parsed = json.loads(match.group()) if match else {}
            routing = parsed.get("routing", "both" if has_image else "rag_only")
            task_plan = parsed.get("task_plan", "자동 계획")
        except Exception:
            routing = "both" if has_image else "rag_only"
            task_plan = "파싱 실패 — 기본 라우팅"
    else:
        routing = "both" if has_image else "rag_only"
        task_plan = "Mock 라우팅"

    if not has_image and routing != "rag_only":
        routing = "rag_only"

    log.append(f"[Orchestrator] 라우팅: {routing}")
    print(f"  [Orchestrator] → {routing}: {task_plan}")
    return {"routing": routing, "task_plan": task_plan, "step_log": log}

def orchestrator_router(state: PipelineState) -> str:
    routing = state.get("routing", "both")
    if routing == "vision_only": return "vision"
    elif routing == "rag_only": return "rag"
    else: return "vision"

print("Orchestrator 노드 정의 완료")


---
## Part 4. Vision Analyst Agent

YOLO → SAM → Depth 순서로 이미지를 분석하고,  
결과를 사람이 읽기 좋은 요약으로 변환합니다.

In [ ]:
# ── Vision Analyst 노드

def vision_analyst_node(state: PipelineState) -> dict:
    """Vision Analyst: YOLO → SAM → Depth → 요약"""
    image_b64 = state.get('image_b64', '')
    log = state.get('step_log', [])
    
    if not image_b64:
        log.append('[Vision] 이미지 없음 — 건너뜀')
        return {'vision_summary': '이미지 없음', 'step_log': log}
    
    # ── Step 1: YOLO 객체 탐지
    yolo_result = yolo.detect(image_b64)
    log.append(f'[Vision/YOLO] {yolo_result["count"]}개 객체 탐지: {yolo_result["classes"]}')
    print(f'  [Vision/YOLO] {yolo_result["count"]}개: {yolo_result["classes"]}')
    
    # ── Step 2: SAM 세그멘테이션
    sam_result = sam.segment(image_b64)
    log.append(f'[Vision/SAM] {sam_result["mask_count"]}개 마스크, 커버리지 {sam_result["coverage_ratio"]}')
    print(f'  [Vision/SAM] 마스크 {sam_result["mask_count"]}개, 커버리지 {sam_result["coverage_ratio"]}')
    
    # ── Step 3: 깊이 추정
    depth_result = depth.estimate(image_b64)
    log.append(f'[Vision/Depth] 평균 깊이: {depth_result["mean_depth"]}m, 근거리: {depth_result["near_objects"]}')
    print(f'  [Vision/Depth] 평균 {depth_result["mean_depth"]}m')
    
    # ── Step 4: 결과 요약 텍스트 생성
    classes_str = ', '.join(yolo_result['classes'])
    near_str = ', '.join(depth_result['near_objects'])
    
    vision_summary = f"""## Vision 분석 결과

### 객체 탐지 (YOLO)
- 탐지된 객체: {yolo_result['count']}개
- 클래스: {classes_str}
- 세부 목록:
{chr(10).join(f"  * {o['class']} (신뢰도 {o['confidence']:.0%})" for o in yolo_result['objects'])}

### 세그멘테이션 (SAM)
- 분할된 영역: {sam_result['mask_count']}개
- 전체 커버리지: {sam_result['coverage_ratio']:.0%}
- 가장 큰 영역: {sam_result['largest_segment']['class']} ({sam_result['largest_segment']['area_ratio']:.0%})

### 깊이 추정 (DepthAnything)
- 평균 깊이: {depth_result['mean_depth']}m
- 깊이 범위: {depth_result['depth_range']['min']}m ~ {depth_result['depth_range']['max']}m
- 근거리 객체: {near_str}
- 원거리 객체: {', '.join(depth_result['far_objects'])}
"""
    
    return {
        'yolo_result': yolo_result,
        'sam_result': sam_result,
        'depth_result': depth_result,
        'vision_summary': vision_summary,
        'step_log': log
    }

print('Vision Analyst 노드 정의 완료')

---
## Part 5. RAG Retriever Agent

질문과 Vision 결과를 바탕으로 관련 지식을 ChromaDB에서 검색합니다.  
Vision 결과가 있으면 탐지된 객체 클래스를 쿼리에 추가하여 더 정확한 검색을 합니다.

In [ ]:
# ── RAG Retriever 노드

def format_rag_context(docs: list, max_chars: int = 2000) -> str:
    """검색 결과 → 컨텍스트 문자열"""
    parts = []
    total = 0
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get('source', 'unknown').split('\\')[-1].split('/')[-1]
        text = f'[문서 {i}: {src}]\n{doc.page_content}'
        if total + len(text) > max_chars:
            break
        parts.append(text)
        total += len(text)
    return '\n\n'.join(parts)

def rag_retriever_node(state: PipelineState) -> dict:
    """RAG Retriever: 질문 + Vision 결과 기반 지식 검색"""
    question = state['question']
    yolo_result = state.get('yolo_result', {})
    log = state.get('step_log', [])
    
    # 쿼리 보강: YOLO 결과가 있으면 탐지된 클래스를 쿼리에 추가
    if yolo_result and yolo_result.get('classes'):
        classes = ' '.join(yolo_result['classes'])
        enhanced_query = f'{question} {classes}'
    else:
        enhanced_query = question
    
    # ChromaDB 검색
    docs = retriever.invoke(enhanced_query)
    context = format_rag_context(docs)
    sources = list({doc.metadata.get('source', '').split('\\')[-1].split('/')[-1] for doc in docs})
    
    log.append(f'[RAG] 쿼리: "{enhanced_query[:50]}..." → {len(docs)}개 청크')
    print(f'  [RAG] {len(docs)}개 검색 | 출처: {sources}')
    
    return {
        'rag_context': context,
        'rag_sources': sources,
        'step_log': log
    }

print('RAG Retriever 노드 정의 완료')

---
## Part 6. Report Writer Agent

Vision 분석 결과 + RAG 컨텍스트를 받아  
**구조화된 최종 분석 리포트**를 생성합니다.

In [ ]:
# -- Report Writer 노드
REPORT_SYSTEM = '''당신은 컴퓨터 비전 분석 전문 리포터입니다.
Vision 분석 결과와 RAG 컨텍스트를 통합하여 구조화된 분석 리포트를 작성합니다.

리포트 형식:
## 분석 요약
- 핵심 발견 사항 2~3가지

## 상세 분석
- 이미지에서 발견된 객체와 의미
- 공간 구성 (깊이/세그멘테이션 기반)

## 배경 지식 연계
- 검색된 지식 베이스 내용과의 연계

## 결론 및 제안
- 사용자 질문에 대한 직접 답변

한국어로 작성하세요.'''

def report_writer_node(state: PipelineState) -> dict:
    question = state["question"]
    vision_summary = state.get("vision_summary", "")
    rag_context = state.get("rag_context", "")
    log = state.get("step_log", [])

    sections = [f"## 사용자 질문\n{question}"]
    if vision_summary and vision_summary != "이미지 없음":
        sections.append(f"## Vision 분석 결과\n{vision_summary}")
    if rag_context:
        sections.append(f"## 관련 지식 (RAG)\n{rag_context[:1500]}")
    combined = "\n\n".join(sections)

    if USE_OLLAMA:
        messages = [SystemMessage(content=REPORT_SYSTEM), HumanMessage(content=combined)]
        report = llm.invoke(messages).content
    else:
        report = f"## 분석 요약\n- Mock 리포트\n\n질문: {question}"

    log.append(f"[Report] {len(report)}자 생성")
    print(f"  [Report Writer] 완료 ({len(report)}자)")
    return {"report": report, "step_log": log}

print("Report Writer 노드 정의 완료")


---
## Part 7. 전체 파이프라인 그래프 조립

4개 에이전트를 LangGraph로 연결합니다.

```
orchestrator
  ├─(vision/both)→ vision_analyst → rag_retriever → report_writer
  └─(rag_only)──→ rag_retriever → report_writer
```

In [ ]:
# ── 전체 파이프라인 그래프 구성

def after_vision_router(state: PipelineState) -> str:
    """Vision 완료 후: routing이 vision_only면 report, 아니면 rag"""
    if state.get('routing') == 'vision_only':
        return 'report'
    return 'rag'

def build_full_pipeline() -> StateGraph:
    """전체 멀티에이전트 파이프라인 그래프 빌드"""
    graph = StateGraph(PipelineState)
    
    # ── 노드 등록
    graph.add_node('orchestrator', orchestrator_node)
    graph.add_node('vision_analyst', vision_analyst_node)
    graph.add_node('rag_retriever', rag_retriever_node)
    graph.add_node('report_writer', report_writer_node)
    
    # ── 진입점
    graph.set_entry_point('orchestrator')
    
    # ── Orchestrator → 조건부 분기
    graph.add_conditional_edges(
        'orchestrator',
        orchestrator_router,
        {
            'vision': 'vision_analyst',
            'rag':    'rag_retriever'
        }
    )
    
    # ── Vision → 조건부 분기 (vision_only vs both)
    graph.add_conditional_edges(
        'vision_analyst',
        after_vision_router,
        {
            'rag':    'rag_retriever',
            'report': 'report_writer'
        }
    )
    
    # ── RAG → Report
    graph.add_edge('rag_retriever', 'report_writer')
    
    # ── Report → END
    graph.add_edge('report_writer', END)
    
    return graph.compile()

pipeline = build_full_pipeline()
print('전체 파이프라인 빌드 완료')

# 그래프 시각화
try:
    from IPython.display import Image, display
    display(Image(pipeline.get_graph().draw_mermaid_png()))
except Exception:
    print('\n그래프 구조 (Mermaid):')
    print(pipeline.get_graph().draw_mermaid())

---
## Part 8. 파이프라인 실행 & 결과 확인

In [ ]:
# ── 파이프라인 실행 래퍼

def run_pipeline(question: str, image_b64: str = None) -> dict:
    """전체 파이프라인 실행"""
    print(f'\n{"="*70}')
    print(f'질문: {question}')
    print(f'이미지: {"있음" if image_b64 else "없음"}')
    print('='*70)
    
    start = time.time()
    
    initial_state: PipelineState = {
        'question': question,
        'image_b64': image_b64,
        'routing': '',
        'task_plan': '',
        'yolo_result': {},
        'sam_result': {},
        'depth_result': {},
        'vision_summary': '',
        'rag_context': '',
        'rag_sources': [],
        'report': '',
        'step_log': [],
        'error': None
    }
    
    result = pipeline.invoke(initial_state)
    elapsed = time.time() - start
    
    print(f'\n{"─"*70}')
    print('실행 로그:')
    for log in result['step_log']:
        print(f'  {log}')
    
    print(f'\n{"─"*70}')
    print('최종 리포트:')
    print(result['report'])
    
    if result['rag_sources']:
        print(f'\n참고 출처: {", ".join(result["rag_sources"])}')
    print(f'\n소요 시간: {elapsed:.2f}초')
    
    return result

print('파이프라인 실행 래퍼 정의 완료')

In [ ]:
# ── 시나리오 1: 이미지 + 질문 (both 라우팅)
result1 = run_pipeline(
    question='이 이미지에서 탐지된 객체들과 YOLO 알고리즘에 대해 설명해주세요',
    image_b64=test_img_b64
)

In [ ]:
# ── 시나리오 2: 텍스트만 (rag_only 라우팅)
result2 = run_pipeline(
    question='LangGraph에서 조건부 엣지(conditional edge)는 어떻게 작동하나요?',
    image_b64=None
)

In [ ]:
# ── 시나리오 3: 이미지 + 깊이 추정 특화 질문
result3 = run_pipeline(
    question='근거리 객체와 원거리 객체의 거리를 분석하고 DepthAnything 모델에 대해 설명해주세요',
    image_b64=test_img_b64
)

---
## Part 9. Streaming 응답 처리

LangGraph의 `stream()` 메서드로 각 노드 실행을 실시간으로 받습니다.  
실제 서비스에서는 이를 SSE(Server-Sent Events) 또는 WebSocket으로 전달합니다.

In [ ]:
# ── LangGraph Streaming (노드 단위 스트리밍)

def run_pipeline_streaming(question: str, image_b64: str = None):
    """스트리밍 모드로 파이프라인 실행 — 각 노드 완료 시점에 결과 출력"""
    print(f'\n스트리밍 실행: "{question}"')
    print('─' * 50)
    
    initial_state: PipelineState = {
        'question': question,
        'image_b64': image_b64,
        'routing': '', 'task_plan': '',
        'yolo_result': {}, 'sam_result': {}, 'depth_result': {},
        'vision_summary': '',
        'rag_context': '', 'rag_sources': [],
        'report': '',
        'step_log': [],
        'error': None
    }
    
    # stream() → 각 노드 완료 시 {node_name: output_dict} 반환
    for event in pipeline.stream(initial_state):
        for node_name, node_output in event.items():
            print(f'\n[{node_name}] 완료')
            
            # 노드별 핵심 출력 표시
            if node_name == 'orchestrator':
                print(f'  라우팅: {node_output.get("routing")}')
                print(f'  계획: {node_output.get("task_plan")}')
            elif node_name == 'vision_analyst':
                yolo = node_output.get('yolo_result', {})
                print(f'  탐지 객체: {yolo.get("count", 0)}개 ({yolo.get("classes", [])})')
            elif node_name == 'rag_retriever':
                print(f'  검색 출처: {node_output.get("rag_sources", [])}')
            elif node_name == 'report_writer':
                report = node_output.get('report', '')
                print(f'  리포트 ({len(report)}자):')
                print(report[:300] + ('...' if len(report) > 300 else ''))
    
    print('\n스트리밍 완료')

run_pipeline_streaming(
    question='이미지의 도로 장면을 분석하고 SAM 세그멘테이션 결과를 설명해주세요',
    image_b64=test_img_b64
)

In [ ]:
# -- Ollama Streaming (토큰 단위 실시간 출력)

def report_writer_streaming(question: str, vision_summary: str, rag_context: str):
    print("\n리포트 스트리밍 생성:")
    print("─" * 50)

    sections = [f"## 사용자 질문\n{question}"]
    if vision_summary and vision_summary != "이미지 없음":
        sections.append(f"## Vision 분석 결과\n{vision_summary[:500]}")
    if rag_context:
        sections.append(f"## 관련 지식\n{rag_context[:400]}")
    combined = "\n\n".join(sections)

    if not USE_OLLAMA:
        print("[Mock] Ollama 없음 — 시뮬레이션")
        for ch in "## 분석 요약\n- Mock 스트리밍 완료":
            print(ch, end="", flush=True)
            time.sleep(0.02)
        print()
        return

    # ChatOllama 스트리밍
    stream_llm = ChatOllama(model=OLLAMA_LLM_MODEL, temperature=0)
    messages = [SystemMessage(content=REPORT_SYSTEM), HumanMessage(content=combined)]
    for chunk in stream_llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print()

# 이전 실행 결과 재활용
report_writer_streaming(
    question="이 이미지의 교통 상황을 분석해주세요",
    vision_summary=result1.get("vision_summary", ""),
    rag_context=result1.get("rag_context", "")
)


---
## Part 10. 파이프라인 평가 & 성능 분석

In [ ]:
# ── 배치 평가: 여러 질문으로 파이프라인 성능 측정
import matplotlib.pyplot as plt

eval_cases = [
    {'question': '이미지에서 차량을 탐지해주세요', 'has_image': True},
    {'question': 'YOLO와 SAM의 차이점은 무엇인가요?', 'has_image': False},
    {'question': '깊이 추정 결과로 가장 가까운 물체는?', 'has_image': True},
    {'question': 'RAG 파이프라인 구성 요소를 설명해주세요', 'has_image': False},
    {'question': '이미지의 세그멘테이션과 DepthAnything 활용법', 'has_image': True},
]

results_eval = []
print('배치 평가 시작...')

for i, case in enumerate(eval_cases, 1):
    start = time.time()
    img = test_img_b64 if case['has_image'] else None
    
    initial: PipelineState = {
        'question': case['question'], 'image_b64': img,
        'routing': '', 'task_plan': '',
        'yolo_result': {}, 'sam_result': {}, 'depth_result': {},
        'vision_summary': '', 'rag_context': '', 'rag_sources': [],
        'report': '', 'step_log': [], 'error': None
    }
    result = pipeline.invoke(initial)
    elapsed = time.time() - start
    
    results_eval.append({
        'question': case['question'][:30] + '...',
        'routing': result.get('routing', 'unknown'),
        'report_len': len(result.get('report', '')),
        'sources': len(result.get('rag_sources', [])),
        'elapsed': round(elapsed, 2)
    })
    print(f'  [{i}/{len(eval_cases)}] {case["question"][:40]}... → {result.get("routing")} ({elapsed:.2f}s)')

print(f'\n배치 평가 완료 ({len(eval_cases)}개 케이스)')

In [ ]:
# ── 평가 결과 시각화
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 그래프 1: 라우팅 분포
routing_counts = {}
for r in results_eval:
    routing_counts[r['routing']] = routing_counts.get(r['routing'], 0) + 1

axes[0].pie(routing_counts.values(), labels=routing_counts.keys(),
            autopct='%1.0f%%', colors=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('라우팅 분포')

# 그래프 2: 케이스별 실행 시간
labels = [f'Q{i+1}' for i in range(len(results_eval))]
times = [r['elapsed'] for r in results_eval]
colors = ['#4C72B0' if r['routing'] == 'both' else
          '#DD8452' if r['routing'] == 'rag_only' else '#55A868'
          for r in results_eval]
axes[1].bar(labels, times, color=colors)
axes[1].set_title('케이스별 실행 시간 (초)')
axes[1].set_ylabel('초')
for i, t in enumerate(times):
    axes[1].text(i, t + 0.01, f'{t:.2f}s', ha='center', va='bottom', fontsize=9)

# 그래프 3: 리포트 길이
report_lens = [r['report_len'] for r in results_eval]
axes[2].bar(labels, report_lens, color='#C44E52')
axes[2].set_title('생성된 리포트 길이 (문자)')
axes[2].set_ylabel('문자 수')

plt.tight_layout()
plt.show()

# 요약 테이블
print('\n평가 요약:')
print(f'{"질문":<32} {"라우팅":<12} {"리포트":<8} {"시간"}')
print('-' * 60)
for r in results_eval:
    print(f'{r["question"]:<32} {r["routing"]:<12} {r["report_len"]:>6}자  {r["elapsed"]:>5}s')

---
## Part 11. Human-in-the-loop (중간 확인 패턴)

Orchestrator의 라우팅 결정을 사람이 확인하고 수정할 수 있는 패턴입니다.  
LangGraph의 `interrupt_before` 기능을 활용합니다.

In [ ]:
# ── Human-in-the-Loop 파이프라인
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer 설정 (Human-in-the-loop에 필수)
checkpointer = MemorySaver()

def build_hitl_pipeline():
    """Human-in-the-loop 파이프라인 (vision_analyst 전 중단)"""
    graph = StateGraph(PipelineState)
    
    graph.add_node('orchestrator', orchestrator_node)
    graph.add_node('vision_analyst', vision_analyst_node)
    graph.add_node('rag_retriever', rag_retriever_node)
    graph.add_node('report_writer', report_writer_node)
    
    graph.set_entry_point('orchestrator')
    graph.add_conditional_edges('orchestrator', orchestrator_router,
                                 {'vision': 'vision_analyst', 'rag': 'rag_retriever'})
    graph.add_conditional_edges('vision_analyst', after_vision_router,
                                 {'rag': 'rag_retriever', 'report': 'report_writer'})
    graph.add_edge('rag_retriever', 'report_writer')
    graph.add_edge('report_writer', END)
    
    # interrupt_before: vision_analyst 실행 전에 일시 중단
    return graph.compile(
        checkpointer=checkpointer,
        interrupt_before=['vision_analyst']
    )

hitl_pipeline = build_hitl_pipeline()
print('HITL 파이프라인 빌드 완료 (interrupt_before: vision_analyst)')

In [ ]:
# ── HITL 실행: 1단계 — Orchestrator까지 실행 후 중단

thread_config = {'configurable': {'thread_id': 'demo-thread-1'}}

initial_hitl: PipelineState = {
    'question': '이미지의 교통 상황을 분석하고 안전 위험 요소를 알려주세요',
    'image_b64': test_img_b64,
    'routing': '', 'task_plan': '',
    'yolo_result': {}, 'sam_result': {}, 'depth_result': {},
    'vision_summary': '', 'rag_context': '', 'rag_sources': [],
    'report': '', 'step_log': [], 'error': None
}

print('1단계: Orchestrator 실행 (vision_analyst 전 중단)...')
for event in hitl_pipeline.stream(initial_hitl, config=thread_config):
    for node_name, output in event.items():
        print(f'  완료: {node_name}')
        if node_name == 'orchestrator':
            print(f'  라우팅 결정: {output.get("routing")}')
            print(f'  계획: {output.get("task_plan")}')

# 현재 상태 확인
current_state = hitl_pipeline.get_state(config=thread_config)
print(f'\n현재 중단 위치: {current_state.next}')
print('→ 사람이 라우팅을 확인하고 수정할 수 있습니다')

In [ ]:
# ── HITL 실행: 2단계 — 사람이 State 수정 후 재개
# 예시: Orchestrator가 'rag_only'로 결정했지만 사람이 'both'로 수정

current_routing = current_state.values.get('routing', '')
print(f'현재 라우팅: {current_routing}')

# 상태 업데이트 (사람 개입)
if current_routing != 'both':
    hitl_pipeline.update_state(
        config=thread_config,
        values={'routing': 'both', 'task_plan': '[사람이 수정] Vision + RAG 모두 실행'},
    )
    print('사람이 라우팅을 "both"로 수정함')
else:
    print('라우팅이 이미 "both" — 수정 불필요')

# 파이프라인 재개 (None을 넘기면 중단된 지점부터 이어서 실행)
print('\n2단계: 파이프라인 재개...')
for event in hitl_pipeline.stream(None, config=thread_config):
    for node_name, output in event.items():
        print(f'  완료: {node_name}')
        if node_name == 'report_writer':
            report = output.get('report', '')
            print(f'  최종 리포트 ({len(report)}자):')
            print(report[:300])

print('\nHITL 파이프라인 완료')

---
## 정리

| 에이전트 | 역할 | 핵심 도구 |
|---------|------|----------|
| `Orchestrator` | 라우팅 결정 | Claude API + 조건부 엣지 |
| `Vision Analyst` | 이미지 분석 | YOLO / SAM / DepthAnything |
| `RAG Retriever` | 지식 검색 | ChromaDB + 쿼리 보강 |
| `Report Writer` | 리포트 생성 | Claude API (Streaming 지원) |

## LangGraph 패턴 요약

| 패턴 | 이 노트북에서 사용된 곳 |
|------|------------------------|
| 조건부 라우팅 | Orchestrator → Vision/RAG 분기 |
| 선형 파이프라인 | RAG → Report |
| Human-in-the-loop | `interrupt_before` + `update_state` |
| Checkpointer | `MemorySaver`로 상태 영속 |
| Streaming | `stream()` + Claude 토큰 스트리밍 |

## 다음 단계: FastAPI 서빙

```python
# api/main.py (예시)
@app.post('/analyze')
async def analyze(image: UploadFile, question: str):
    image_b64 = base64.b64encode(await image.read()).decode()
    result = pipeline.invoke({'question': question, 'image_b64': image_b64, ...})
    return {'report': result['report']}
```